# 🏐 Binsu Star — Volleyball Match Analyzer

Runs on **Google Colab (free GPU)**. It:
1. Downloads a match video (yt-dlp)
2. Detects **players** and the **ball** with YOLOv8 + tracking
3. Builds the **ball trajectory** and finds **touch** moments
4. Gives a first-pass **action guess** (serve / receive / set / spike)
5. Builds a JSON **report** and **POSTs it to your Cloudflare Worker**

### Be honest about what this does
- ✅ **Reliable:** ball & player *detection*, tracking, ball *trajectory*.
- ⚠️ **Approximate:** *action labels* (serve/receive/set/spike) and *player identity*. Out of the box these are **heuristics**, not a trained volleyball model — treat them as a draft. The last section shows how to **train** a proper model on labelled clips to make them accurate.

> Set the runtime to GPU first: **Runtime → Change runtime type → T4 GPU**.


## 1) Install dependencies

In [ ]:
!pip -q install ultralytics yt-dlp opencv-python-headless requests numpy
import os, json, time, subprocess, math
import numpy as np
print("ready")

## 2) Config — edit these

In [ ]:
# --- videos to analyse (your links) ---
VIDEO_URLS = [
    "https://youtu.be/SnAr01djvYk",
    "https://youtu.be/110ud26-1UU",
    "https://youtu.be/SMCFIvtXAEk",
]
# the "last game" you want the report for goes last / analyse it on its own:
LAST_GAME_URL = "https://youtu.be/7EAPWRJlt6E"

# --- videos for the (optional) training section ---
TRAIN_URLS = [
    "https://youtu.be/AxeDAYpeqZY",
    "https://www.youtube.com/live/VwyOa81R7us",
    "https://youtu.be/7EAPWRJlt6E",
]

# --- your Cloudflare Worker (site + backend) ---
CF_ENDPOINT = "https://first-test-2021.binsustar.workers.dev"
ADMIN_KEY   = ""   # <-- paste your admin password (64928 by default) to push the report

# --- optional: map track IDs -> real names later; leave empty for now ---
ROSTER = []        # e.g. ["Coas", "Qarz", "Veldora"]

GAME_LABEL   = "Binsu Star — Last Game"
ANALYSE_URL  = LAST_GAME_URL      # which video this run analyses
MAX_SECONDS  = 240                # cap processing time; raise for a full match
SAMPLE_FPS   = 5                  # frames per second to analyse (lower = faster)
print("config set — analysing:", ANALYSE_URL)

## 3) Download the video

In [ ]:
def download(url, out="game.mp4"):
    if os.path.exists(out):
        os.remove(out)
    # 720p mp4 keeps it light; yt-dlp handles normal + live VODs
    cmd = ["yt-dlp", "-f", "best[height<=720][ext=mp4]/best[height<=720]/best",
           "--merge-output-format", "mp4", "-o", out, url]
    subprocess.run(cmd, check=True)
    return out

video_path = download(ANALYSE_URL)
print("downloaded:", video_path, os.path.getsize(video_path)//1_000_000, "MB")

## 4) Detect players + ball, with tracking
YOLOv8 (COCO): class **0 = person**, **32 = sports ball**. `model.track` keeps a stable ID per player across frames.

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")   # small+fast; swap to yolov8s.pt / yolov8m.pt for more accuracy

cap = cv2.VideoCapture(video_path)
vid_fps = cap.get(cv2.CAP_PROP_FPS) or 30
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
step = max(1, int(round(vid_fps / SAMPLE_FPS)))
max_frames = int(MAX_SECONDS * vid_fps)
print(f"video {W}x{H} @ {vid_fps:.1f}fps — sampling every {step} frames")

ball_track = []      # (t_sec, x, y)
player_frames = []   # per sampled frame: list of (track_id, cx, cy)
seen_players = set()

fi = 0
while True:
    ok, frame = cap.read()
    if not ok or fi > max_frames:
        break
    if fi % step == 0:
        t = fi / vid_fps
        res = model.track(frame, classes=[0, 32], persist=True, tracker="bytetrack.yaml", verbose=False)[0]
        players = []
        if res.boxes is not None:
            for b in res.boxes:
                cls = int(b.cls[0]); x1,y1,x2,y2 = b.xyxy[0].tolist()
                cx, cy = (x1+x2)/2, (y1+y2)/2
                if cls == 32:                       # ball
                    ball_track.append((t, cx, cy))
                elif cls == 0:                      # player
                    tid = int(b.id[0]) if b.id is not None else -1
                    players.append((tid, cx, cy)); seen_players.add(tid)
        player_frames.append((t, players))
    fi += 1
cap.release()
print(f"sampled {len(player_frames)} frames | ball seen in {len(ball_track)} | distinct players tracked: {len(seen_players)}")

## 5) Ball trajectory + touch detection
A *touch* is a local low/turn point in the ball path near a player — the moment someone contacts the ball. (Heuristic.)

In [ ]:
# smooth ball path and find contacts (direction changes in vertical velocity)
traj = [(t,x,y) for (t,x,y) in ball_track]
touches = []   # (t, x, y, nearest_player_id)

def nearest_player(t, x, y):
    best, bd = -1, 1e9
    for (ft, players) in player_frames:
        if abs(ft - t) > 0.4:  # within ~0.4s
            continue
        for (tid, px, py) in players:
            d = (px-x)**2 + (py-y)**2
            if d < bd: bd, best = d, tid
    return best

for i in range(2, len(traj)-2):
    t0,x0,y0 = traj[i-2]; t1,x1,y1 = traj[i]; t2,x2,y2 = traj[i+2]
    v_before = (y1 - y0); v_after = (y2 - y1)
    # a contact ~ vertical velocity flips (ball was falling then rises, or vice-versa)
    if v_before * v_after < 0 and abs(v_before) + abs(v_after) > 4:
        touches.append((t1, x1, y1, nearest_player(t1, x1, y1)))

# de-duplicate touches that are <0.3s apart
dedup = []
for tch in touches:
    if not dedup or tch[0] - dedup[-1][0] > 0.3:
        dedup.append(tch)
touches = dedup
print("detected touches:", len(touches))

## 6) First-pass action guess (heuristic)
⚠️ These labels are **rough** — court zone + ball height/speed only. Replace with the trained model (last section) for real accuracy.

In [ ]:
def classify(tch, prev_t):
    t, x, y, pid = tch
    top = y < H * 0.33          # ball high in frame
    low = y > H * 0.66          # ball low
    gap = (t - prev_t) if prev_t else 99
    if gap > 3.0:               # long pause before contact -> new rally
        return "serve"
    if low:
        return "receive"        # low dig-height contact
    if top:
        return "spike"          # high contact
    return "set"                # mid-height contact

events, prev_t = [], None
counts = {"serve":0,"receive":0,"set":0,"spike":0}
for tch in touches:
    a = classify(tch, prev_t); prev_t = tch[0]
    counts[a] += 1
    events.append({"t": round(tch[0],2), "action": a, "player_track_id": tch[3],
                   "ball_xy": [round(tch[1]), round(tch[2])]})
print("action counts (heuristic):", counts)

## 7) Build the report

In [ ]:
def name_for(tid):
    if tid is None or tid < 0: return f"Unknown"
    if ROSTER: return ROSTER[tid % len(ROSTER)]   # crude map until you label identities
    return f"Player #{tid}"

per_player = {}
for e in events:
    nm = name_for(e["player_track_id"])
    d = per_player.setdefault(nm, {"serve":0,"receive":0,"set":0,"spike":0,"touches":0})
    d[e["action"]] += 1; d["touches"] += 1

report = {
    "video": ANALYSE_URL,
    "frames_sampled": len(player_frames),
    "players_tracked": len(seen_players),
    "ball_points": len(traj),
    "touches": len(touches),
    "action_counts": counts,
    "per_player": per_player,
    "events": events[:500],
    "trajectory": [[round(t,2), round(x), round(y)] for (t,x,y) in traj[:2000]],
    "notes": "Detection & trajectory are measured; action labels and player identities are heuristic and need a trained model + labels for accuracy.",
}
print(json.dumps({k:v for k,v in report.items() if k not in ("events","trajectory")}, indent=2))
open("report.json","w").write(json.dumps(report))

## 8) Push the report to Cloudflare
Stores it in your Worker's KV via `/admin/analysis` (needs your admin key). The site can then read `/analyses` and `/analysis?id=`.

In [ ]:
import requests
if not ADMIN_KEY:
    print("Set ADMIN_KEY in the config cell to push. Skipping — report saved locally as report.json")
else:
    r = requests.post(CF_ENDPOINT + "/admin/analysis",
                      headers={"Content-Type":"application/json","X-Admin-Key":ADMIN_KEY},
                      json={"label": GAME_LABEL, "report": report}, timeout=30)
    print(r.status_code, r.text[:300])

## 9) Run it multiple times / on every video
Good practice: run the last game a few times (or bump `SAMPLE_FPS`) and compare — heuristic labels are noisy, so averaging across runs is more trustworthy.

In [ ]:
# quick loop over all your videos (detection + trajectory + heuristic actions)
def analyse(url, label):
    global ANALYSE_URL, GAME_LABEL
    ANALYSE_URL, GAME_LABEL = url, label
    print("\n=== ", label, url, " ===")
    # re-run cells 3-7 manually, or refactor them into functions and call here.
    # left as a loop stub so you control cost on the free GPU.
for u in VIDEO_URLS:
    print("queued:", u)
print("Tip: run cells 3-8 per video, changing ANALYSE_URL, to build several reports.")

## 10) (Optional) Train a real volleyball model
The heuristic above won't be accurate. To make **serve/receive/set/spike** and **player identity** real, you need **labelled clips**:

1. **Label** a few hundred contacts. Easiest: use the **tagging analyzer on your website** (players + serve/receive/set/spike + ball path) — that produces the labels for free, OR label frames in **Roboflow**.
2. Export a YOLO dataset (`data.yaml`) with your classes (e.g. `serve, receive, set, spike, ball`).
3. Fine-tune on this free GPU:


In [ ]:
# after you have a labelled dataset (Roboflow export or your own data.yaml):
# from ultralytics import YOLO
# m = YOLO("yolov8s.pt")
# m.train(data="/content/dataset/data.yaml", epochs=80, imgsz=960, batch=16)
# m.val()
# best = "runs/detect/train/weights/best.pt"
# ...then load `best` in cell 4 instead of yolov8n.pt for accurate actions.
print("Training needs labelled data first — see step 1. Detection+trajectory work now; accurate actions need this step.")